# BESOIN 4 : MODELE DE PREDICTION DE LA PUISSANCE NOMINALE (DOUAE)


# 1. Importation des bibliothèques et configuration

In [3]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")




# 2. Chargement des données

In [4]:
df = pd.read_csv("data/export_IA.csv", low_memory=False)


def categoriser_puissance(p):
    if p <= 7.4:
        return "Lente (<= 7.4 kW)"
    elif p <= 22:
        return "Normale (7.4 - 22 kW)"
    elif p <= 50:
        return "Acceleree (22 - 50 kW)"
    elif p <= 150:
        return "Rapide (50 - 150 kW)"
    else:
        return "Ultra-rapide (> 150 kW)"

TARGET = 'puissance_categorie'
df[TARGET] = df['puissance'].apply(categoriser_puissance)

print("Distribution de la cible :")
print(df[TARGET].value_counts())


FEATURES = ['implantation', 'nb_pdc', 'prise_ccs', 'prise_chademo',
            'prise_type2', 'prise_ef', 'gratuit', 'condition_acces']

data = df[FEATURES + [TARGET]].copy()
data = data.dropna(subset=[TARGET])
print(f"\nShape apres filtre cible : {data.shape}")

Distribution de la cible :
puissance_categorie
Normale (7.4 - 22 kW)      61791
Lente (<= 7.4 kW)          26856
Acceleree (22 - 50 kW)     18683
Ultra-rapide (> 150 kW)    17966
Rapide (50 - 150 kW)       13638
Name: count, dtype: int64

Shape apres filtre cible : (138934, 9)


## 2.1 Nettoyage, Imputation et Encodage des données

In [5]:
#  Nettoyage et encodage

bool_cols = ['prise_ccs', 'prise_chademo', 'prise_type2', 'prise_ef', 'gratuit']
for col in bool_cols:
    data[col] = data[col].map({True: 1, False: 0, 'True': 1, 'False': 0,
                                'TRUE': 1, 'FALSE': 0}).fillna(0).astype(int)

# Encodage implantation
data['implantation'] = data['implantation'].fillna('inconnu')
le_implantation = LabelEncoder()
data['implantation'] = le_implantation.fit_transform(data['implantation'])
joblib.dump(le_implantation, 'fichier_pkl/le_implantation_b4.pkl')

# Encodage condition_acces
data['condition_acces'] = data['condition_acces'].str.strip().fillna('inconnu')
le_acces = LabelEncoder()
data['condition_acces'] = le_acces.fit_transform(data['condition_acces'])
joblib.dump(le_acces, 'fichier_pkl/le_acces_b4.pkl')

# Imputation nb_pdc
data['nb_pdc'] = data['nb_pdc'].fillna(data['nb_pdc'].median())

# 3. Analyse Exploratoire des Données (Visualisations)

In [6]:
print("\nGeneration des graphiques de justification...")

# Graphique 1 : Distribution des tranches de puissance
plt.figure(figsize=(10, 5))
ordre = ["Lente (<= 7.4 kW)", "Normale (7.4 - 22 kW)", "Acceleree (22 - 50 kW)",
         "Rapide (50 - 150 kW)", "Ultra-rapide (> 150 kW)"]
counts = df[TARGET].value_counts().reindex(ordre)
sns.barplot(x=counts.index, y=counts.values, palette='Set2')
plt.title("Distribution des tranches de puissance nominale", fontsize=12, fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.ylabel("Nombre de bornes")
plt.tight_layout()
plt.savefig("figures/distribution_puissance.png", dpi=300)
plt.close()

# Graphique 2 : Proportion de prises CCS par tranche de puissance
plt.figure(figsize=(10, 5))
temp = df.copy()
temp[TARGET] = temp['puissance'].apply(categoriser_puissance)
temp['prise_ccs'] = temp['prise_ccs'].map({True: 1, False: 0, 'True': 1, 'False': 0}).fillna(0)
sns.barplot(data=temp, x=TARGET, y='prise_ccs', order=ordre, errorbar=None, palette='Blues_r')
plt.title("Proportion de prises CCS par tranche de puissance", fontsize=12, fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.ylabel("Proportion")
plt.tight_layout()
plt.savefig("figures/justification_prise_ccs_b4.png", dpi=300)
plt.close()


Generation des graphiques de justification...


C:\Users\HI\AppData\Local\Temp\ipykernel_23212\4231257577.py:8: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=counts.index, y=counts.values, palette='Set2')
C:\Users\HI\AppData\Local\Temp\ipykernel_23212\4231257577.py:21: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=temp, x=TARGET, y='prise_ccs', order=ordre, errorbar=None, palette='Blues_r')


# 4. Préparation des données pour le Machine Learning

## 4.1 Séparation des données (Train/Test Split)

In [7]:

X = data[FEATURES].astype(float)
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain : {X_train.shape[0]}  |  Test : {X_test.shape[0]}")


Train : 111147  |  Test : 27787


## 4.2 Normalisation des fonctionnalités (Scaling) et export du pipeline

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

joblib.dump(scaler, 'fichier_pkl/scaler_pretraitement_b4.pkl')
joblib.dump(list(FEATURES), 'fichier_pkl/features_b4.pkl')

['fichier_pkl/features_b4.pkl']

# 5. Entraînement et optimisation du modèle (GridSearchCV)

In [9]:
print("\nRecherche des meilleurs hyperparametres (Regression Logistique)")

param_grid = {
    'C':        [0.1, 1.0, 10.0],
    'solver':   ['lbfgs', 'saga'],
    'max_iter': [300]
}

grid_search = GridSearchCV(
    estimator=LogisticRegression(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

meilleur_modele = grid_search.best_estimator_
print(f"Meilleurs hyperparametres : {grid_search.best_params_}")


Recherche des meilleurs hyperparametres (Regression Logistique)
Meilleurs hyperparametres : {'C': 1.0, 'max_iter': 300, 'solver': 'lbfgs'}


# 6. Évaluation des performances du modèle

In [10]:
y_pred = meilleur_modele.predict(X_test_scaled)


print("     RAPPORT D'EVALUATION DE LA CLASSIFICATION")
print(classification_report(y_test, y_pred))


     RAPPORT D'EVALUATION DE LA CLASSIFICATION
                         precision    recall  f1-score   support

 Acceleree (22 - 50 kW)       0.68      0.26      0.37      3737
      Lente (<= 7.4 kW)       0.75      0.40      0.52      5371
  Normale (7.4 - 22 kW)       0.68      0.93      0.78     12358
   Rapide (50 - 150 kW)       0.57      0.17      0.26      2728
Ultra-rapide (> 150 kW)       0.56      0.89      0.69      3593

               accuracy                           0.66     27787
              macro avg       0.65      0.53      0.52     27787
           weighted avg       0.66      0.66      0.61     27787



## 6.1 Matrice de confusion

In [11]:
matrice = confusion_matrix(y_test, y_pred, labels=ordre)
plt.figure(figsize=(9, 7))
sns.heatmap(matrice, annot=True, fmt='d', cmap='Blues',
            xticklabels=ordre, yticklabels=ordre)
plt.title("Matrice de Confusion - Puissance Nominale", fontsize=12, fontweight='bold')
plt.ylabel('Realite Terrain')
plt.xlabel('Prediction Modele')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig("figures/matrice_confusion_b4.png", dpi=300)
plt.close()

# 7. Sauvegarde du modèle prédictif final

In [12]:
joblib.dump(meilleur_modele, 'fichier_pkl/modele_classification_b4.pkl')


['fichier_pkl/modele_classification_b4.pkl']